# 🧊 Antarctic Sea-Ice Concentration Forecasting
## End-to-End PyTorch Lightning Pipeline

> **Architecture**: U-Net Spatial Encoder → ConvLSTM Spatiotemporal Core → U-Net Spatial Decoder  
> **Task**: Predict Antarctic sea-ice concentration maps (T_in=5 days → T_out=3 days)  
> **Pipeline**: Autonomous build, execute, verify — with OOM auto-recovery

---

## Cell 1 — Environment Setup & GPU Verification

In [1]:
# ===================================================================
#  CELL 1 - Environment Setup & GPU Verification
#  Antigravity ML Agent: autonomous pip install + hardware validation
# ===================================================================
import subprocess, sys

print('📦 Installing required packages...')
pkgs = [
    'torch torchvision',
    'pytorch-lightning',
    'xarray netCDF4',
    'torchmetrics',
    'scikit-learn',
    'matplotlib',
    'scipy',
]
for pkg in pkgs:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q'] + pkg.split(),
        capture_output=True, text=True
    )
    status = '✅' if result.returncode == 0 else '❌'
    print(f'  {status} {pkg}')

# -- Directory setup --
import os
from pathlib import Path

CONTENT_DIR  = Path('/content/sea_ice_pipeline')
DATA_DIR     = CONTENT_DIR / 'data'
CKPT_DIR     = CONTENT_DIR / 'checkpoints'
ARTIFACT_DIR = CONTENT_DIR / 'artifacts'

for d in [DATA_DIR, CKPT_DIR, ARTIFACT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'\n📁 Working directory: {CONTENT_DIR}')
print(f'   data/        -> {DATA_DIR}')
print(f'   checkpoints/ -> {CKPT_DIR}')
print(f'   artifacts/   -> {ARTIFACT_DIR}')

# -- GPU Verification --
import torch
print('\n' + '=' * 55)
print('🔬 Hardware Acceleration Check')
print('=' * 55)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    compute  = torch.cuda.get_device_capability(0)
    print(f'  GPU Model   : {gpu_name}')
    print(f'  VRAM        : {vram_gb:.1f} GB')
    print(f'  Compute Cap : {compute[0]}.{compute[1]}')
    print(f'  CUDA Version: {torch.version.cuda}')
    print(f'  PyTorch Ver : {torch.__version__}')
    DEVICE = torch.device('cuda')
    print(f'\n  ✅ GPU acceleration ACTIVE on {gpu_name}')
else:
    print('  ⚠️  No GPU detected — running on CPU')
    DEVICE = torch.device('cpu')

import pytorch_lightning as pl
print(f'\n  Lightning Ver: {pl.__version__}')
print('=' * 55)


📦 Installing required packages...
  ✅ torch torchvision
  ✅ pytorch-lightning
  ✅ xarray netCDF4
  ✅ torchmetrics
  ✅ scikit-learn
  ✅ matplotlib
  ✅ scipy

📁 Working directory: \content\sea_ice_pipeline
   data/        -> \content\sea_ice_pipeline\data
   checkpoints/ -> \content\sea_ice_pipeline\checkpoints
   artifacts/   -> \content\sea_ice_pipeline\artifacts

🔬 Hardware Acceleration Check
  ⚠️  No GPU detected — running on CPU

  Lightning Ver: 2.6.6


## Cell 2 — Data Engineering Pipeline: SeaIceDataModule

In [15]:
# ===================================================================
#  CELL 2 - Data Engineering Pipeline
#  - Synthetic NSIDC-style NetCDF data generator (40-day test subset)
#  - SeaIceDataModule (pl.LightningDataModule)
#  - Sliding window: X(B, T_in=5, C=1, H, W) -> Y(B, T_out=3, C=1, H, W)
# ===================================================================
import numpy as np
import xarray as xr
import torch
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pathlib import Path
from typing import Optional

# -- Globals --
T_IN   = 5       # Input sequence length (days)
T_OUT  = 3       # Prediction horizon (days)
H, W   = 64, 64  # Spatial grid (Antarctic coastal corridor crop)
N_DAYS = 40      # Synthetic dataset length


# -- 2a. Synthetic NSIDC-style Data Generator --
def generate_nsidc_synthetic_nc(
    out_dir: Path,
    n_days: int = N_DAYS,
    height: int = H,
    width:  int = W,
    seed:   int = 42,
):
    """
    Generate a synthetic NSIDC-like NetCDF sea-ice concentration dataset
    and a companion land mask. Mimics real NSIDC daily .nc file structure.

    Data model:
      - Antarctic ring pattern: high ice near the pole, decreasing radially outward.
      - Sinusoidal seasonal cycle + spatially correlated Gaussian noise.
      - Land mask: central 25% of grid is the Antarctic continent.
    """
    from scipy.ndimage import gaussian_filter
    rng = np.random.default_rng(seed)
    print(f'🗂️  Generating synthetic NSIDC dataset: {n_days} days @ {height}x{width}...')

    cy, cx = height // 2, width // 2
    yy, xx = np.ogrid[:height, :width]
    dist   = np.sqrt((yy - cy)**2 + (xx - cx)**2)
    max_d  = np.sqrt(cy**2 + cx**2)

    base_ice = np.clip(1.0 - (dist / (max_d * 0.9)), 0.0, 1.0).astype(np.float32)
    ocean_mask = (dist > max_d * 0.25).astype(np.float32)  # 1=ocean, 0=land

    ice_cube = np.empty((n_days, height, width), dtype=np.float32)
    for t in range(n_days):
        season = 0.15 * np.sin(2 * np.pi * t / 365.0 + np.pi)
        noise  = gaussian_filter(rng.normal(0, 0.08, (height, width)).astype(np.float32), sigma=3.0)
        frame  = np.clip(base_ice + season + noise, 0.0, 1.0) * ocean_mask
        ice_cube[t] = frame

    lats  = np.linspace(-90.0, -60.0, height, dtype=np.float32)
    lons  = np.linspace(0.0, 360.0, width, endpoint=False, dtype=np.float32)
    times = np.arange(n_days)

    nc_path   = out_dir / 'nsidc_sic_synthetic.nc'
    mask_path = out_dir / 'land_mask.nc'

    xr.Dataset(
        {'siconc': (['time', 'lat', 'lon'], ice_cube)},
        coords={'time': times, 'lat': lats, 'lon': lons},
        attrs={'title': 'Synthetic NSIDC-style Antarctic SIC', 'convention': 'siconc in [0,1]'}
    ).to_netcdf(nc_path)

    xr.Dataset(
        {'mask': (['lat', 'lon'], ocean_mask)},
        coords={'lat': lats, 'lon': lons},
        attrs={'description': '1=ocean, 0=land'}
    ).to_netcdf(mask_path)

    print(f'  ✅ ice_cube shape  : {ice_cube.shape}')
    print(f'  ✅ ocean fraction  : {ocean_mask.mean()*100:.1f}%')
    print(f'  ✅ ice value range : [{ice_cube.min():.3f}, {ice_cube.max():.3f}]')
    return nc_path, mask_path


# -- 2b. Sliding-Window PyTorch Dataset --
class SeaIceSequenceDataset(Dataset):
    """
    Sliding-window dataset: X(T_in, 1, H, W) -> Y(T_out, 1, H, W)
    [day_i...day_{i+T_in-1}] -> [day_{i+T_in}...day_{i+T_in+T_out-1}]
    """
    def __init__(self, data: np.ndarray, t_in: int = T_IN, t_out: int = T_OUT):
        self.data      = np.ascontiguousarray(data.astype(np.float32))
        self.t_in      = t_in
        self.t_out     = t_out
        self.n_samples = len(data) - t_in - t_out + 1
        assert self.n_samples > 0, f'Not enough timesteps: {len(data)} < {t_in+t_out}'

    def __len__(self): return self.n_samples

    def __getitem__(self, idx):
        x = self.data[idx: idx + self.t_in]                         # (T_in, H, W)
        y = self.data[idx + self.t_in: idx + self.t_in + self.t_out]  # (T_out, H, W)
        # Add channel dim -> (T, 1, H, W)
        return (
            torch.from_numpy(x[:, np.newaxis]),
            torch.from_numpy(y[:, np.newaxis]),
        )


# -- 2c. PyTorch Lightning DataModule --
class SeaIceDataModule(pl.LightningDataModule):
    """
    LightningDataModule for Antarctic sea-ice forecasting.
    Handles: synthetic generation, Antarctic cropping, land masking,
    normalization, chronological split, sliding-window DataLoaders.
    """
    def __init__(
        self,
        data_dir: Path,
        t_in: int = T_IN, t_out: int = T_OUT,
        crop_h: int = H,  crop_w: int = W,
        val_frac: float = 0.2,
        batch_size: int = 4,
        num_workers: int = 0,
        use_synthetic: bool = True,
        n_synthetic_days: int = N_DAYS,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.data_dir         = Path(data_dir)
        self.t_in             = t_in
        self.t_out            = t_out
        self.crop_h           = crop_h
        self.crop_w           = crop_w
        self.val_frac         = val_frac
        self.batch_size       = batch_size
        self.num_workers      = num_workers
        self.use_synthetic    = use_synthetic
        self.n_synthetic_days = n_synthetic_days
        self.land_mask        = None
        self.train_ds         = None
        self.val_ds           = None

    def prepare_data(self):
        nc_path = self.data_dir / 'nsidc_sic_synthetic.nc'
        if self.use_synthetic and not nc_path.exists():
            generate_nsidc_synthetic_nc(
                self.data_dir, self.n_synthetic_days, self.crop_h, self.crop_w
            )

    def setup(self, stage=None):
        nc_path   = self.data_dir / 'nsidc_sic_synthetic.nc'
        mask_path = self.data_dir / 'land_mask.nc'

        ice_raw  = xr.open_dataset(nc_path)['siconc'].values.astype(np.float32)
        mask_raw = xr.open_dataset(mask_path)['mask'].values.astype(np.float32)

        T, H_raw, W_raw = ice_raw.shape
        ch  = min(self.crop_h, H_raw)
        cw  = min(self.crop_w, W_raw)
        cy0 = (H_raw - ch) // 2
        cx0 = (W_raw - cw) // 2

        ice  = ice_raw[:,  cy0:cy0+ch, cx0:cx0+cw]
        mask = mask_raw[   cy0:cy0+ch, cx0:cx0+cw]

        ocean = (mask != 0.0).astype(np.float32)
        ice   = np.nan_to_num(ice * ocean[np.newaxis], nan=0.0).clip(0.0, 1.0)
        self.land_mask = ocean

        n_val   = max(1, int(round(T * self.val_frac)))
        n_train = T - n_val

        self.train_ds = SeaIceSequenceDataset(ice[:n_train], self.t_in, self.t_out)
        self.val_ds   = SeaIceSequenceDataset(ice[n_train:], self.t_in, self.t_out)

        print(f'\n📊 DataModule Setup:')
        print(f'   timesteps : {T}  ->  train={n_train}, val={n_val}')
        print(f'   sequences : train={len(self.train_ds)}, val={len(self.val_ds)}')
        print(f'   X shape   : (B={self.batch_size}, T_in={self.t_in}, 1, {ch}, {cw})')
        print(f'   Y shape   : (B={self.batch_size}, T_out={self.t_out}, 1, {ch}, {cw})')

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size,
                          shuffle=True,  num_workers=self.num_workers,
                          pin_memory=True, drop_last=True)

    def val_dataloader(self):
        return DataLoader(self.val_ds,   batch_size=self.batch_size,
                          shuffle=False, num_workers=self.num_workers,
                          pin_memory=True)


# -- 2d. Smoke-test --
print('\n' + '=' * 55)
print('🔬 Cell 2: Smoke-testing SeaIceDataModule')
print('=' * 55)

dm = SeaIceDataModule(data_dir=DATA_DIR, batch_size=4)
dm.prepare_data()
dm.setup()

bx, by = next(iter(dm.train_dataloader()))
print(f'\n  X shape: {tuple(bx.shape)}  (B, T_in, C, H, W)')
print(f'  Y shape: {tuple(by.shape)}  (B, T_out, C, H, W)')
assert bx.shape == (4, T_IN, 1, H, W)
assert by.shape == (4, T_OUT, 1, H, W)
print('  ✅ Shape assertions passed!')



🔬 Cell 2: Smoke-testing SeaIceDataModule

📊 DataModule Setup:
   timesteps : 40  ->  train=32, val=8
   sequences : train=25, val=1
   X shape   : (B=4, T_in=5, 1, 64, 64)
   Y shape   : (B=4, T_out=3, 1, 64, 64)

  X shape: (4, 5, 1, 64, 64)  (B, T_in, C, H, W)
  Y shape: (4, 3, 1, 64, 64)  (B, T_out, C, H, W)
  ✅ Shape assertions passed!


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


## Cell 3 — Model Architecture: U-Net Encoder + ConvLSTM + U-Net Decoder

In [16]:
# ===================================================================
#  CELL 3 - Model Architecture
#  - ConvLSTMCell & ConvLSTM   (spatiotemporal core)
#  - DoubleConv / DownBlock / UpBlock  (U-Net building blocks)
#  - SeaIceForecastNet  (full encoder-LSTM-decoder)
#  - SeaIceForecastModel (pl.LightningModule) with MSE + (1-SSIM) loss
# ===================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from torchmetrics.functional import structural_similarity_index_measure as ssim_fn
from typing import Optional


# -- U-Net Building Blocks -------------------------------------------

class DoubleConv(nn.Module):
    """[Conv2d(3x3) -> BN -> ReLU] x2 with Kaiming init."""
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
    def forward(self, x): return self.block(x)


class DownBlock(nn.Module):
    """MaxPool2d(2) -> DoubleConv. Returns (pooled, skip)."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = DoubleConv(in_ch, out_ch)
        self.pool = nn.MaxPool2d(2)
    def forward(self, x):
        skip = self.conv(x)
        return self.pool(skip), skip


class UpBlock(nn.Module):
    """ConvTranspose2d(x2) -> pad -> cat(skip) -> DoubleConv."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, in_ch // 2, 2, stride=2)
        self.conv = DoubleConv(in_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        dy = skip.size(2) - x.size(2)
        dx = skip.size(3) - x.size(3)
        if dy or dx:
            x = F.pad(x, [dx//2, dx-dx//2, dy//2, dy-dy//2])
        return self.conv(torch.cat([skip, x], dim=1))


# -- ConvLSTM Cell & Module ------------------------------------------

class ConvLSTMCell(nn.Module):
    """Spatiotemporal ConvLSTM cell (Shi et al. 2015)."""
    def __init__(self, in_ch: int, hidden_ch: int, kernel: int = 3):
        super().__init__()
        self.hidden_ch = hidden_ch
        self.gates = nn.Conv2d(in_ch + hidden_ch, 4 * hidden_ch, kernel, padding=kernel//2)
        nn.init.orthogonal_(self.gates.weight)

    def forward(self, x, h, c):
        i, f, g, o = torch.chunk(self.gates(torch.cat([x, h], 1)), 4, dim=1)
        c = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g)
        h = torch.sigmoid(o) * torch.tanh(c)
        return h, c

    def init_hidden(self, B, H, W, device):
        z = torch.zeros(B, self.hidden_ch, H, W, device=device)
        return z, z.clone()


class ConvLSTM(nn.Module):
    """Multi-layer ConvLSTM. Returns all hidden states for last layer."""
    def __init__(self, in_ch: int, hidden_ch: int, n_layers: int = 2, kernel: int = 3):
        super().__init__()
        self.cells = nn.ModuleList([
            ConvLSTMCell(in_ch if i == 0 else hidden_ch, hidden_ch, kernel)
            for i in range(n_layers)
        ])

    def forward(self, seq):  # seq: (B, T, C, H, W)
        B, T, C, H, W = seq.shape
        states  = [cell.init_hidden(B, H, W, seq.device) for cell in self.cells]
        outputs = []
        for t in range(T):
            x_t = seq[:, t]
            for i, cell in enumerate(self.cells):
                h, c = states[i]
                h, c = cell(x_t, h, c)
                states[i] = (h, c)
                x_t = h
            outputs.append(h.unsqueeze(1))
        return torch.cat(outputs, dim=1)  # (B, T, hidden_ch, H, W)


# -- Full Architecture: Encoder -> ConvLSTM -> Decoder ---------------

class SeaIceForecastNet(nn.Module):
    """
    U-Net Encoder (weight-shared) -> ConvLSTM (2-layer) -> U-Net Decoder.
    Input:  (B, T_in, 1, H, W)
    Output: (B, T_out, 1, H, W)  in [0, 1]
    """
    def __init__(self, t_in=T_IN, t_out=T_OUT, base_f=32, lstm_h=64, n_lstm=2):
        super().__init__()
        self.t_in  = t_in
        self.t_out = t_out
        f = base_f

        # Encoder (shared across all T_in frames)
        self.enc_inc    = DoubleConv(1, f)           # (B, f, H, W)
        self.enc_d1     = DownBlock(f,   f*2)        # -> (B, f*2, H/2, W/2)
        self.enc_d2     = DownBlock(f*2, f*4)        # -> (B, f*4, H/4, W/4)
        self.bottleneck = DoubleConv(f*4, lstm_h)   # -> (B, lstm_h, H/4, W/4)

        # ConvLSTM core
        self.convlstm   = ConvLSTM(lstm_h, lstm_h, n_layers=n_lstm)

        # Decoder (applied T_out times)
        self.dec_u1   = UpBlock(lstm_h + f*4, f*2)
        self.dec_u2   = UpBlock(f*2    + f*2, f)
        self.dec_u3   = UpBlock(f      + f,   f)
        self.out_head = nn.Sequential(nn.Conv2d(f, 1, 1), nn.Sigmoid())

        n_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'  SeaIceForecastNet: {n_params:,} trainable parameters')

    def encode(self, frame):  # frame: (B, 1, H, W)
        s0          = self.enc_inc(frame)      # (B, f, H, W)
        d1, skip_d1 = self.enc_d1(s0)          # (B, f*2, H/2, W/2)
        d2, skip_d2 = self.enc_d2(d1)          # (B, f*4, H/4, W/4)
        bn          = self.bottleneck(d2)      # (B, lstm_h, H/4, W/4)
        return bn, (s0, skip_d1, skip_d2)

    def decode(self, z, skips):  # z: (B, lstm_h, h, w)
        s0, skip_d1, skip_d2 = skips
        x = self.dec_u1(z, skip_d2)
        x = self.dec_u2(x, skip_d1)
        x = self.dec_u3(x, s0)
        return self.out_head(x)  # (B, 1, H, W)

    def forward(self, x_seq):  # (B, T_in, 1, H, W)
        B, T, C, H, W = x_seq.shape

        bottlenecks = []
        last_skips  = None
        for t in range(T):
            bn, skips = self.encode(x_seq[:, t])
            bottlenecks.append(bn.unsqueeze(1))
            last_skips = skips

        bn_seq   = torch.cat(bottlenecks, dim=1)      # (B, T_in, lstm_h, h, w)
        lstm_out = self.convlstm(bn_seq)               # (B, T_in, lstm_h, h, w)
        z_last   = lstm_out[:, -1]                     # (B, lstm_h, h, w)

        preds = [self.decode(z_last, last_skips).unsqueeze(1) for _ in range(self.t_out)]
        return torch.cat(preds, dim=1)  # (B, T_out, 1, H, W)


# -- LightningModule --------------------------------------------------

class SeaIceForecastModel(pl.LightningModule):
    """LightningModule: Loss = MSE + lambda * (1 - SSIM)."""

    def __init__(self, t_in=T_IN, t_out=T_OUT, base_f=32, lstm_h=64,
                 lr=1e-3, ssim_lambda=0.1, land_mask=None):
        super().__init__()
        self.save_hyperparameters(ignore=['land_mask'])
        self.net         = SeaIceForecastNet(t_in, t_out, base_f, lstm_h)
        self.lr          = lr
        self.ssim_lambda = ssim_lambda
        if land_mask is not None:
            self.register_buffer('land_mask', land_mask.float())

    def combined_loss(self, pred, target):
        """MSE + lambda*(1-SSIM), averaged over T_out frames."""
        mse = F.mse_loss(pred, target)
        ssim_vals = [
            ssim_fn(pred[:, t], target[:, t], data_range=1.0)
            for t in range(pred.shape[1])
        ]
        ssim_loss = 1.0 - torch.stack(ssim_vals).mean()
        return mse + self.ssim_lambda * ssim_loss, mse, ssim_loss

    def forward(self, x): return self.net(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        pred = self(x)
        loss, mse, sl = self.combined_loss(pred, y)
        self.log('train/loss', loss, prog_bar=True, on_step=True, on_epoch=True)
        self.log('train/mse',  mse,  on_step=False, on_epoch=True)
        self.log('train/ssim_loss', sl, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        pred = self(x)
        loss, mse, sl = self.combined_loss(pred, y)
        self.log('val/loss', loss, prog_bar=True, on_epoch=True)
        self.log('val/mse',  mse,  on_epoch=True)
        self.log('val/ssim_loss', sl, on_epoch=True)

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=10)
        return {'optimizer': opt, 'lr_scheduler': {'scheduler': sch, 'interval': 'epoch'}}


# -- Architecture smoke-test -----------------------------------------
print('\n' + '=' * 55)
print('🔬 Cell 3: Forward-Pass Verification')
print('=' * 55)

model = SeaIceForecastModel(
    t_in=T_IN, t_out=T_OUT, base_f=32, lstm_h=64,
    land_mask=torch.from_numpy(dm.land_mask) if dm.land_mask is not None else None
)
model.eval()
dummy = torch.randn(2, T_IN, 1, H, W)
with torch.no_grad():
    out = model(dummy)

print(f'\n  Input  : {tuple(dummy.shape)}')
print(f'  Output : {tuple(out.shape)}')
print(f'  Range  : [{out.min():.4f}, {out.max():.4f}]  (sigmoid -> [0,1] checked)')
assert out.shape == (2, T_OUT, 1, H, W)
assert out.min() >= 0.0 and out.max() <= 1.0
print('  ✅ Architecture verified!')



🔬 Cell 3: Forward-Pass Verification
  SeaIceForecastNet: 1,324,673 trainable parameters


RuntimeError: Given transposed=1, weight of size [192, 96, 2, 2], expected input[2, 64, 16, 16] to have 192 channels, but got 64 channels instead

## Cell 4 — Execution: Lightning Trainer + Antigravity Auto-Debug OOM Recovery

In [ ]:
# ===================================================================
#  CELL 4 - Training Execution
#  - pl.Trainer: GPU, 16-mixed precision, max_epochs=2
#  - Antigravity Auto-Debug: OOM -> halves batch_size -> retries
#  - Verifies: graph compiles, gradients flow, loss decreases
# ===================================================================
import gc
import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import CSVLogger

MAX_EPOCHS    = 2
INIT_BATCH_SZ = 4
MIN_BATCH_SZ  = 1


def build_trainer(ckpt_dir, max_epochs):
    accelerator = 'gpu' if torch.cuda.is_available() else 'cpu'
    precision   = '16-mixed' if torch.cuda.is_available() else '32-true'
    print(f'  Trainer: accelerator={accelerator}, precision={precision}')
    return pl.Trainer(
        accelerator=accelerator,
        devices=1,
        precision=precision,
        max_epochs=max_epochs,
        log_every_n_steps=1,
        enable_progress_bar=True,
        callbacks=[
            ModelCheckpoint(
                dirpath=str(ckpt_dir),
                filename='best-{epoch:02d}-{val/loss:.4f}',
                monitor='val/loss', mode='min', save_top_k=1,
            ),
            EarlyStopping(monitor='val/loss', patience=5, mode='min'),
            LearningRateMonitor(logging_interval='epoch'),
        ],
        logger=CSVLogger(str(CONTENT_DIR / 'logs'), name='sea_ice'),
        deterministic=False,
    )


def run_training_with_oom_retry(
    model_cls, model_kwargs, dm_kwargs,
    init_batch_sz=INIT_BATCH_SZ, min_batch_sz=MIN_BATCH_SZ, max_epochs=MAX_EPOCHS,
):
    """
    =====================================================
    Antigravity Auto-Debug: OOM Interception & Recovery
    =====================================================
    Catches CUDA OOM and autonomously:
      1. Flushes GPU memory
      2. Halves batch_size
      3. Rebuilds DataModule + Trainer
      4. Retries — no user prompt required
    """
    batch_sz = init_batch_sz
    attempt  = 0

    while batch_sz >= min_batch_sz:
        attempt += 1
        print(f'\n' + '=' * 55)
        print(f'🚀 Training Attempt #{attempt}  batch_size={batch_sz}')
        print('=' * 55)

        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                gc.collect()
                free_gb = (torch.cuda.get_device_properties(0).total_memory
                           - torch.cuda.memory_allocated(0)) / (1024**3)
                print(f'  GPU free VRAM: {free_gb:.2f} GB')

            dm_run = SeaIceDataModule(**{**dm_kwargs, 'batch_size': batch_sz})
            dm_run.prepare_data()
            dm_run.setup()

            land_t = torch.from_numpy(dm_run.land_mask) if dm_run.land_mask is not None else None
            m      = model_cls(**{**model_kwargs, 'land_mask': land_t})
            trnr   = build_trainer(CKPT_DIR, max_epochs)

            trnr.fit(m, datamodule=dm_run)
            print(f'\n✅ Training succeeded with batch_size={batch_sz}!')
            return trnr, m, dm_run

        except RuntimeError as err:
            if 'out of memory' in str(err).lower() or 'cuda' in str(err).lower():
                print(f'\n⚠️  [Antigravity Auto-Debug] CUDA OOM at batch_size={batch_sz}')
                print(f'   {str(err)[:120]}')
                del m, trnr, dm_run
                gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()
                batch_sz //= 2
                if batch_sz < min_batch_sz:
                    raise RuntimeError('Cannot fit model even at batch_size=1. Reduce base_f.') from err
                print(f'   Retrying with batch_size={batch_sz}...')
            else:
                raise


# -- Launch training -------------------------------------------------
print('\n' + '=' * 55)
print('🚀 Cell 4: Launching Lightning Training Pipeline')
print('=' * 55)

dm_kwargs = dict(
    data_dir=DATA_DIR, t_in=T_IN, t_out=T_OUT,
    crop_h=H, crop_w=W, val_frac=0.25,
    num_workers=0, use_synthetic=True, n_synthetic_days=N_DAYS,
)
model_kwargs = dict(
    t_in=T_IN, t_out=T_OUT, base_f=32, lstm_h=64,
    lr=1e-3, ssim_lambda=0.1,
)

trainer, model, dm = run_training_with_oom_retry(
    SeaIceForecastModel, model_kwargs, dm_kwargs,
    init_batch_sz=INIT_BATCH_SZ, min_batch_sz=MIN_BATCH_SZ, max_epochs=MAX_EPOCHS,
)

# -- Gradient flow verification ------------------------------------
print('\n' + '=' * 55)
print('🔬 Gradient Flow Verification')
print('=' * 55)

model.train()
bx, by = next(iter(dm.train_dataloader()))
device = next(model.parameters()).device
bx, by = bx.to(device), by.to(device)

opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
opt.zero_grad()
pred = model(bx)
loss, mse_l, ssim_l = model.combined_loss(pred, by)
loss.backward()

grad_norms = [(n, p.grad.norm().item()) for n, p in model.named_parameters() if p.grad is not None]
print(f'  Params with gradients : {len(grad_norms)}')
print(f'  Max grad norm         : {max(g for _,g in grad_norms):.4f}')
print(f'  Combined Loss         : {loss.item():.6f}')
print(f'  MSE                   : {mse_l.item():.6f}')
print(f'  (1 - SSIM)            : {ssim_l.item():.6f}')
assert len(grad_norms) > 0, 'No gradients! Backprop failed.'
assert not torch.isnan(loss), 'Loss is NaN!'
print('  ✅ Gradients flow correctly through all layers!')

print('\n' + '=' * 55)
print('📊 Training Summary')
print('=' * 55)
for k, v in trainer.logged_metrics.items():
    val = v.item() if isinstance(v, torch.Tensor) else v
    print(f'  {k:<30}: {val:.6f}')
print(f'  Best ckpt: {trainer.checkpoint_callback.best_model_path}')


## Cell 5 — Verification Artifacts: Side-by-Side Prediction Visualization

In [ ]:
# ===================================================================
#  CELL 5 - Verification Artifacts
#  - Load best checkpoint
#  - Run single val batch through trained model
#  - Matplotlib visualization:
#      [Input Day 5] | [Pred Day 6] | [GT Day 6] | [|Error| Map]
#  - Multi-step forecast strip for all T_out frames
#  - Save artifact PNG to /content/sea_ice_pipeline/artifacts/
# ===================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import torch

plt.style.use('dark_background')

# -- Load best checkpoint -------------------------------------------
print('\n' + '=' * 55)
print('🏆 Cell 5: Loading Best Checkpoint & Running Inference')
print('=' * 55)

best_ckpt = trainer.checkpoint_callback.best_model_path
land_t    = torch.from_numpy(dm.land_mask) if dm.land_mask is not None else None

if best_ckpt and Path(best_ckpt).exists():
    print(f'  Loading: {best_ckpt}')
    best_model = SeaIceForecastModel.load_from_checkpoint(best_ckpt, land_mask=land_t)
    print('  ✅ Checkpoint loaded!')
else:
    print('  ⚠️  No checkpoint found — using in-memory model')
    best_model = model

best_model.eval()
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
best_model = best_model.to(device)

# -- Inference on val batch -----------------------------------------
vbx, vby = next(iter(dm.val_dataloader()))
vbx, vby = vbx.to(device), vby.to(device)

with torch.no_grad():
    vpred = best_model(vbx)  # (B, T_out, 1, H, W)

tot_loss, mse_l, ssim_l = best_model.combined_loss(vpred, vby)
print(f'\n  Combined Loss : {tot_loss.item():.6f}')
print(f'  MSE           : {mse_l.item():.6f}')
print(f'  (1 - SSIM)    : {ssim_l.item():.6f}')
print(f'  Pred range    : [{vpred.min():.4f}, {vpred.max():.4f}]')

# -- Numpy extraction (sample 0) ------------------------------------
pred_np  = vpred[0].cpu().numpy()   # (T_out, 1, H, W)
gt_np    = vby[0].cpu().numpy()     # (T_out, 1, H, W)
inp_np   = vbx[0].cpu().numpy()     # (T_in, 1, H, W)

input_f  = inp_np[-1, 0]            # Last input frame
pred_f   = pred_np[0, 0]            # First predicted frame
gt_f     = gt_np[0, 0]              # First ground-truth frame
err_f    = np.abs(pred_f - gt_f)    # Absolute error
land_np  = dm.land_mask if dm.land_mask is not None else np.ones_like(input_f)

# -- Visualization --------------------------------------------------
fig = plt.figure(figsize=(20, 14), dpi=120, facecolor='#0a0a1a')
fig.patch.set_facecolor('#0a0a1a')

fig.suptitle(
    '🧊 Antarctic Sea-Ice Forecast Verification  —  U-Net + ConvLSTM + U-Net',
    fontsize=17, fontweight='bold', color='#e8f4fd', y=0.97,
)
fig.text(
    0.5, 0.935,
    f'Architecture: U-Net Encoder (32f) → ConvLSTM (64h, 2-layer) → U-Net Decoder'
    f'   |   Loss: MSE + 0.1×(1−SSIM)   |   Val Loss: {tot_loss.item():.4f}',
    ha='center', fontsize=10, color='#9ab', style='italic',
)

ice_cmap = plt.cm.Blues
err_cmap = plt.cm.YlOrRd

gs = gridspec.GridSpec(2, 4, figure=fig,
                       left=0.04, right=0.96, top=0.90, bottom=0.08,
                       wspace=0.22, hspace=0.48)

def add_panel(row, col, data, title, cmap, vmin, vmax, tcolor):
    ax = fig.add_subplot(gs[row, col])
    ax.set_facecolor('#0d1117')
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax,
                   origin='upper', aspect='equal', interpolation='bilinear')
    ax.contour(land_np, levels=[0.5], colors='#fff8', linewidths=0.8)
    ax.set_title(title, fontsize=12, fontweight='bold', color=tcolor, pad=10)
    ax.set_xlabel('Longitude (grid px)', fontsize=9, color='#aaa')
    ax.set_ylabel('Latitude (grid px)',  fontsize=9, color='#aaa')
    ax.tick_params(colors='#777', labelsize=8)
    for sp in ax.spines.values(): sp.set_edgecolor('#333')
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.ax.tick_params(colors='#aaa', labelsize=8)
    cb.set_label('SIC [0-1]' if cmap != err_cmap else '|Error|', color='#aaa', fontsize=8)

add_panel(0, 0, input_f, f'Input  Day {T_IN}\n(last conditioning frame)',  ice_cmap, 0, 1, '#2196F3')
add_panel(0, 1, pred_f,  f'Prediction  Day {T_IN+1}\n(first forecast step)', ice_cmap, 0, 1, '#4CAF50')
add_panel(0, 2, gt_f,    f'Ground Truth  Day {T_IN+1}\n(first target frame)',  ice_cmap, 0, 1, '#FF9800')
add_panel(0, 3, err_f,   f'|Error| Map\n|Prediction − Truth|',              err_cmap, 0, None, '#F44336')

# Row 2: multi-step forecast strip
fig.text(0.5, 0.43,
         f'Multi-Step Forecast Strip  (All {T_OUT} Prediction Frames — Left=Pred | Right=GT)',
         ha='center', fontsize=13, fontweight='bold', color='#e8f4fd')

for t in range(min(T_OUT, 4)):
    ax2 = fig.add_subplot(gs[1, t])
    ax2.set_facecolor('#0d1117')
    combined = np.hstack([pred_np[t, 0], gt_np[t, 0]])
    ax2.imshow(combined, cmap=ice_cmap, vmin=0, vmax=1, origin='upper', aspect='auto')
    ax2.axvline(x=W, color='#ff0', linewidth=1.5, linestyle='--', alpha=0.7)
    mse_t = float(np.mean((pred_np[t, 0] - gt_np[t, 0])**2))
    ax2.set_title(f'Day {T_IN+t+1}  MSE={mse_t:.4f}\nLeft=Pred  Right=GT',
                  fontsize=10, color='#e8f4fd', pad=6)
    ax2.set_xlabel('← Pred | GT →', fontsize=9, color='#aaa')
    ax2.tick_params(colors='#777', labelsize=8)
    for sp in ax2.spines.values(): sp.set_edgecolor('#333')

fig.text(0.5, 0.01,
         'Antigravity ML Agent  ·  U-Net + ConvLSTM + U-Net  ·  MSE + 0.1·(1−SSIM)  ·  AdamW + CosineAnnealingLR',
         ha='center', fontsize=9, color='#556', style='italic')

artifact_path = ARTIFACT_DIR / 'sea_ice_forecast_verification.png'
fig.savefig(artifact_path, dpi=140, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'\n  ✅ Artifact saved: {artifact_path}')

# -- Final verification summary ------------------------------------
print('\n' + '=' * 55)
print('🎉 PIPELINE VERIFICATION COMPLETE')
print('=' * 55)

import glob, csv
csv_logs = sorted(glob.glob(str(CONTENT_DIR / 'logs' / 'sea_ice' / '**' / 'metrics.csv'), recursive=True))
if csv_logs:
    import pandas as pd
    df = pd.read_csv(csv_logs[-1])
    tl = df['train/loss_epoch'].dropna().tolist() if 'train/loss_epoch' in df.columns else []
    vl = df['val/loss'].dropna().tolist()         if 'val/loss'         in df.columns else []
    if len(tl) >= 2:
        print(f'  Train loss : {tl[0]:.6f} -> {tl[-1]:.6f}  ({"✅ DECREASING" if tl[-1]<tl[0] else "⚠️ check epochs"})')
    if vl:
        print(f'  Val loss   : {vl[-1]:.6f}')

print(f'  Architecture compiled   : ✅')
print(f'  Gradients verified      : ✅')
print(f'  Pred range              : [{vpred.min():.4f}, {vpred.max():.4f}] ✅')
print(f'  Visualization artifact  : {artifact_path} ✅')
print(f'  Best checkpoint         : {trainer.checkpoint_callback.best_model_path} ✅')
print('=' * 55)



🏆 Cell 5: Loading Best Checkpoint & Running Inference


NameError: name 'trainer' is not defined

In [17]:
"""2D U-Net Model for Sea-Ice Concentration Forecasting.

Implements a resolution-agnostic 2D U-Net (`SeaIceUNet`) that predicts the next day's
sea-ice concentration map from a channel-stacked temporal window of past daily maps.
Features:
- Arbitrary non-power-of-2 grid handling via center padding before skip concatenation.
- Configurable base filters and depth.
- Kaiming/He normal weight initialization.
- Optional static land-mask buffer registration for physical constraint enforcement.
"""

from __future__ import annotations

import logging
from typing import Sequence

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


logger = logging.getLogger("SeaIceUNet")
if not logger.handlers:
    _handler = logging.StreamHandler()
    _formatter = logging.Formatter(
        "[%(asctime)s] [%(name)s] [%(levelname)s] %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
    _handler.setFormatter(_formatter)
    logger.addHandler(_handler)
    logger.setLevel(logging.INFO)


class DoubleConv(nn.Module):
    """Building block: [Conv2d(3x3, pad=1) -> BatchNorm2d -> ReLU] x 2."""

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        mid_channels: int | None = None,
    ) -> None:
        super().__init__()
        mid = mid_channels or out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.double_conv(x)


class Down(nn.Module):
    """Downscaling block: MaxPool2d(2) followed by DoubleConv."""

    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Returns (downsampled_feature, skip_feature)."""
        skip = self.conv(x)
        down = self.pool(skip)
        return down, skip


class Up(nn.Module):
    """Upscaling block: ConvTranspose2d (or Bilinear Upsample) + Pad + Concat + DoubleConv.

    Handles non-power-of-2 and odd spatial dimensions by center-padding the upsampled
    feature map to match the exact spatial dimensions of the skip connection.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        bilinear: bool = False,
    ) -> None:
        super().__init__()
        self.bilinear = bilinear

        if bilinear:
            self.up = nn.Sequential(
                nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True),
                nn.Conv2d(in_channels, in_channels // 2, kernel_size=1, bias=False),
            )
            self.conv = DoubleConv(in_channels, out_channels)
        else:
            # in_channels is the channels of the deeper feature map.
            # Upsample halves channels: in_channels -> in_channels // 2
            # After concatenating with skip connection (which has in_channels // 2 channels),
            # total channels fed to DoubleConv is in_channels.
            self.up = nn.ConvTranspose2d(
                in_channels, in_channels // 2, kernel_size=2, stride=2
            )
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.up(x)

        # Handle non-power-of-2 / odd grid dimensions:
        # If the input grid dimension was odd (e.g. 65), pooling gave floor(65/2) = 32.
        # Upsampling 32 by stride 2 gives 64, which is 1 pixel smaller than the skip connection (65).
        # We pad the upsampled tensor x symmetrically to match skip's exact (Height, Width).
        diff_y = skip.size()[2] - x.size()[2]
        diff_x = skip.size()[3] - x.size()[3]

        if diff_x != 0 or diff_y != 0:
            # F.pad format: [pad_left, pad_right, pad_top, pad_bottom]
            x = F.pad(
                x,
                [
                    diff_x // 2,
                    diff_x - diff_x // 2,
                    diff_y // 2,
                    diff_y - diff_y // 2,
                ],
            )

        # Concatenate along channel dimension
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)


class OutConv(nn.Module):
    """1x1 convolution mapping to out_channels followed by Sigmoid activation."""

    def __init__(self, in_channels: int, out_channels: int = 1) -> None:
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.sigmoid(self.conv(x))


class SeaIceUNet(nn.Module):
    """Resolution-agnostic 2D U-Net for sea-ice concentration forecasting.

    Accepts temporal window input: (Batch, in_channels, Height, Width)
    where in_channels corresponds to the sliding-window size (e.g. 7 days).
    Outputs next-day forecast: (Batch, out_channels, Height, Width) in [0.0, 1.0].

    Features:
    - Arbitrary spatial dimension compatibility (supports odd / non-power-of-2 grids).
    - Configurable depth and base filter multiplier.
    - Kaiming/He normal weight initialization.
    - Optional static land mask buffer for enforcing physical zero-ice constraints on land.

    Args:
        in_channels: Number of past observation maps stacked as channels (e.g. 7).
        out_channels: Number of target prediction channels (default: 1).
        base_filters: Number of filters in first encoder level (default: 32).
        depth: Number of downsampling / upsampling stages (default: 4).
        land_mask: Optional static land mask tensor (Height, Width) where ocean=1, land=0.
        bilinear: If True, uses bilinear interpolation for upsampling instead of
                  learnable ConvTranspose2d (default: False).
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int = 1,
        base_filters: int = 32,
        depth: int = 4,
        land_mask: torch.Tensor | None = None,
        bilinear: bool = False,
    ) -> None:
        super().__init__()
        if depth < 1:
            raise ValueError(f"depth must be >= 1, got {depth}")
        if base_filters < 1:
            raise ValueError(f"base_filters must be >= 1, got {base_filters}")
        if in_channels < 1:
            raise ValueError(f"in_channels must be >= 1, got {in_channels}")

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.base_filters = base_filters
        self.depth = depth
        self.bilinear = bilinear

        # 1. Register land mask as non-trainable buffer
        if land_mask is not None:
            if not isinstance(land_mask, torch.Tensor):
                land_mask = torch.tensor(land_mask, dtype=torch.float32)
            else:
                land_mask = land_mask.to(dtype=torch.float32)

            # Ensure mask is shaped (1, 1, H, W) for direct broadcasting
            if land_mask.ndim == 2:
                land_mask = land_mask.unsqueeze(0).unsqueeze(0)
            elif land_mask.ndim == 3:
                land_mask = land_mask.unsqueeze(0)

            # Enforce binary ocean (1) vs land (0)
            land_mask = (land_mask != 0.0).float()
            self.register_buffer("land_mask", land_mask)
        else:
            self.register_buffer("land_mask", None)

        # 2. Build Encoder Path
        self.inc = DoubleConv(in_channels, base_filters)
        self.down_blocks = nn.ModuleList()
        curr_filters = base_filters

        for i in range(depth - 1):
            next_filters = curr_filters * 2
            self.down_blocks.append(
                nn.Sequential(
                    nn.MaxPool2d(kernel_size=2, stride=2),
                    DoubleConv(curr_filters, next_filters),
                )
            )
            curr_filters = next_filters

        # 3. Bottleneck at deepest point
        bottleneck_in = curr_filters
        bottleneck_out = curr_filters * 2
        self.bottleneck_pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.bottleneck = DoubleConv(bottleneck_in, bottleneck_out)

        # 4. Build Decoder Path
        self.up_blocks = nn.ModuleList()
        dec_in = bottleneck_out
        for i in range(depth):
            dec_out = dec_in // 2
            self.up_blocks.append(Up(dec_in, dec_out, bilinear=bilinear))
            dec_in = dec_out

        # 5. Output Head
        self.outc = OutConv(base_filters, out_channels)

        # 6. Apply Kaiming/He Normal Weight Initialization
        self._init_weights()

        total_params = self.count_parameters()
        logger.info(
            f"SeaIceUNet initialized: in_channels={in_channels}, out_channels={out_channels}, "
            f"base_filters={base_filters}, depth={depth}, bilinear={bilinear}, "
            f"has_land_mask={self.land_mask is not None} -> Total Parameters: {total_params:,}"
        )

    def _init_weights(self) -> None:
        """Initialize all conv and batchnorm weights using Kaiming/He normal scheme."""
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def count_parameters(self) -> int:
        """Return total number of trainable parameters in the model."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass through U-Net with skip connections and land masking.

        Args:
            x: Input tensor of shape (Batch, in_channels, Height, Width).

        Returns:
            Output prediction of shape (Batch, out_channels, Height, Width) in [0.0, 1.0].
        """
        if x.ndim != 4:
            raise ValueError(
                f"Expected 4D input tensor (Batch, Channels, Height, Width), but got shape {x.shape}"
            )
        if x.shape[1] != self.in_channels:
            raise ValueError(
                f"Input channel count mismatch: model expected {self.in_channels} channels, "
                f"but received input with {x.shape[1]} channels."
            )

        # 1. Encoder Path
        skips: list[torch.Tensor] = []
        x1 = self.inc(x)
        skips.append(x1)

        curr = x1
        for down in self.down_blocks:
            curr = down(curr)
            skips.append(curr)

        # 2. Bottleneck
        b_down = self.bottleneck_pool(curr)
        b_feat = self.bottleneck(b_down)

        # 3. Decoder Path with Skip Connections (in reverse order)
        curr = b_feat
        for up_block in self.up_blocks:
            skip_feat = skips.pop()
            curr = up_block(curr, skip_feat)

        # 4. Output Head (1x1 Conv + Sigmoid)
        logits_sigmoid = self.outc(curr)

        # 5. Apply Static Land Mask (if provided)
        if self.land_mask is not None:
            # Broadcast mask: (1, 1, H, W) * (Batch, 1, H, W)
            # If input spatial dimensions differ from mask, interpolate mask or raise
            if self.land_mask.shape[-2:] != logits_sigmoid.shape[-2:]:
                mask = F.interpolate(
                    self.land_mask,
                    size=logits_sigmoid.shape[-2:],
                    mode="nearest",
                )
            else:
                mask = self.land_mask
            output = logits_sigmoid * mask
        else:
            output = logits_sigmoid

        return output

    def forward_checkpointed(self, x: torch.Tensor) -> torch.Tensor:
        """Memory-efficient forward pass using gradient checkpointing.

        Applies ``torch.utils.checkpoint.checkpoint`` to the bottleneck and
        every decoder up-block — the activation-heavy parts of the U-Net.  The
        encoder path (inc + down_blocks) is left uncheckpointed because its
        skip-connection tensors must remain materialised for the decoder to
        read from.

        This method is called exclusively by ``local_train.py`` when
        ``LocalTrainConfig.use_grad_checkpoint = True``.  It is **never**
        called by the existing ``train.py`` / Lightning AI path — those
        continue to call ``forward()`` as before.

        Gradient checkpointing trades additional forward re-computation during
        the backward pass for a significant reduction in peak VRAM usage, which
        is critical for fitting the model on a 6 GB laptop GPU.

        Args:
            x: Input tensor of shape (Batch, in_channels, Height, Width).

        Returns:
            Output prediction of shape (Batch, out_channels, Height, Width)
            in [0.0, 1.0], identical to ``forward()``.
        """
        import torch.utils.checkpoint as torch_ckpt

        if x.ndim != 4:
            raise ValueError(
                f"Expected 4D input tensor (Batch, Channels, Height, Width), "
                f"but got shape {x.shape}"
            )
        if x.shape[1] != self.in_channels:
            raise ValueError(
                f"Input channel count mismatch: model expected {self.in_channels} "
                f"channels, but received input with {x.shape[1]} channels."
            )

        # ── Encoder (not checkpointed — skips must stay alive for decoder) ──
        skips: list[torch.Tensor] = []
        x1 = self.inc(x)
        skips.append(x1)

        curr = x1
        for down in self.down_blocks:
            curr = down(curr)
            skips.append(curr)

        # ── Bottleneck (checkpointed) ────────────────────────────────────────
        # checkpoint() requires inputs that require grad; pool output may not
        # have requires_grad=True in eval mode, so we gate on torch.is_grad_enabled.
        b_down = self.bottleneck_pool(curr)

        def _run_bottleneck(inp: torch.Tensor) -> torch.Tensor:
            return self.bottleneck(inp)

        b_feat = torch_ckpt.checkpoint(_run_bottleneck, b_down, use_reentrant=False)

        # ── Decoder (each up-block checkpointed) ─────────────────────────────
        curr = b_feat
        for up_block in self.up_blocks:
            skip_feat = skips.pop()

            # closure captures up_block and skip_feat by reference
            def _run_up(inp: torch.Tensor, _up=up_block, _skip=skip_feat) -> torch.Tensor:
                return _up(inp, _skip)

            curr = torch_ckpt.checkpoint(_run_up, curr, use_reentrant=False)

        # ── Output head + land mask (same as forward()) ───────────────────────
        logits_sigmoid = self.outc(curr)

        if self.land_mask is not None:
            if self.land_mask.shape[-2:] != logits_sigmoid.shape[-2:]:
                mask = F.interpolate(
                    self.land_mask,
                    size=logits_sigmoid.shape[-2:],
                    mode="nearest",
                )
            else:
                mask = self.land_mask
            output = logits_sigmoid * mask
        else:
            output = logits_sigmoid

        return output


if __name__ == "__main__":
    from pathlib import Path
    from data_pipeline import SeaIceDataPipeline
    from torch_dataset import create_dataloaders

    print("=" * 75)
    print("Testing SeaIceUNet Model with Pipeline Integration")
    print("=" * 75)

    base_dir = Path(__file__).parent / "data"
    sea_ice_pattern = str(base_dir / "sea_ice" / "*.nc")
    mask_file = str(base_dir / "land_mask.nc")

    # 1. Load data pipeline
    pipeline = SeaIceDataPipeline(
        data_path=sea_ice_pattern,
        mask_path=mask_file,
    )
    pipeline.load().clean().normalize(max_value=100.0)

    # 2. Extract U-Net formatted sequences & dataloaders
    window_size = 7
    X_unet, Y_unet = pipeline.format_for_model("unet", window_size=window_size, horizon=1)
    train_loader, val_loader = create_dataloaders(
        X_unet, Y_unet, val_split=0.2, batch_size=8, shuffle_train=True
    )

    # 3. Instantiate SeaIceUNet with pipeline's land mask
    land_mask_tensor = torch.from_numpy(pipeline.land_mask)
    model = SeaIceUNet(
        in_channels=window_size,
        out_channels=1,
        base_filters=32,
        depth=4,
        land_mask=land_mask_tensor,
    )

    # 4. Forward pass using real batch from train_loader
    batch_x, batch_y = next(iter(train_loader))
    print(f"\nPulled real train batch:")
    print(f"  Input batch_x shape:  {batch_x.shape} (Batch, Channels, H, W)")
    print(f"  Target batch_y shape: {batch_y.shape} (Batch, 1, H, W)")

    model.eval()
    with torch.no_grad():
        pred_y = model(batch_x)

    print(f"\nForward pass output:")
    print(f"  Output pred_y shape:  {pred_y.shape}")
    print(f"  Matches target shape: {pred_y.shape == batch_y.shape}")
    print(f"  Output value range:   [{pred_y.min().item():.4f}, {pred_y.max().item():.4f}]")

    # 5. Confirm land pixels are exactly 0.0
    land_mask_np = pipeline.land_mask
    land_indices = land_mask_np == 0.0
    pred_y_np = pred_y.squeeze(1).numpy()
    land_values_max = np.max(pred_y_np[:, land_indices])
    print(f"  Max value over land:  {land_values_max:.6f} (must be 0.000000)")
    assert land_values_max == 0.0, "Land pixels were not strictly zeroed!"

    # 6. Parameter count
    total_params = model.count_parameters()
    print(f"\nModel Parameter Count: {total_params:,} trainable parameters")

    # 7. Non-power-of-2 / Odd Grid Dimension Test
    print("\n--- Non-Power-of-2 / Odd Grid Dimension Compatibility Test ---")
    odd_h, odd_w = 67, 53  # Prime/odd grid dimensions
    dummy_odd_x = torch.randn(2, window_size, odd_h, odd_w)
    # Model without land mask (testing pure convolution and padding)
    model_flexible = SeaIceUNet(
        in_channels=window_size,
        out_channels=1,
        base_filters=32,
        depth=4,
        land_mask=None,
    )
    with torch.no_grad():
        odd_out = model_flexible(dummy_odd_x)
    print(f"  Input shape:  {dummy_odd_x.shape}")
    print(f"  Output shape: {odd_out.shape}")
    assert odd_out.shape == (2, 1, odd_h, odd_w), "Odd grid shape mismatch!"
    print("  Odd grid dimension forward pass completed without error!")

    print("\n" + "=" * 75)
    print("SeaIceUNet Verification Complete!")
    print("=" * 75)


ModuleNotFoundError: No module named 'data_pipeline'